In [18]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer
from sklearn.preprocessing import MinMaxScaler
from langdetect import detect

plt.rcParams["figure.figsize"] = (20, 13)
%matplotlib inline
%config InlineBackend.figure_format = "retina"

In [19]:
interactions = pd.read_csv("../data_final_project/KuaiRec/data/big_matrix.csv")
small_interactions = pd.read_csv("../data_final_project/KuaiRec/data/small_matrix.csv")
item_features = pd.read_csv("../data_final_project/KuaiRec/data/item_daily_features.csv")
captions = pd.read_csv("../data_final_project/KuaiRec/data/kuairec_caption_category.csv", lineterminator='\n')

def clean_df(df):
    df = df.dropna()
    df = df.drop_duplicates()  
    return df  

def clean_df_timestamp(df):
    df = clean_df(df)
    df = df[df["timestamp"] >= 0]
    return df

captions = clean_df(captions)
captions = captions.drop_duplicates(subset='video_id')

item_features = clean_df(item_features)
item_features = item_features.drop_duplicates(subset='video_id')

train_df = clean_df_timestamp(interactions)
test_df = clean_df_timestamp(small_interactions)

In [20]:
item_features['upload_dt'] = pd.to_datetime(item_features['upload_dt'])
item_features['date'] = pd.to_datetime(item_features['date'])
item_features['video_age'] = (item_features['date'] - item_features['upload_dt']).dt.days
item_features['is_short_video'] = (item_features['video_duration'].fillna(0) <= 30).astype(int)

In [21]:
correlation = item_features[[
       'video_duration', 'video_width',
       'video_height', 'music_id',
       'show_cnt', 'show_user_num', 'play_cnt', 'play_user_num',
       'play_duration', 'complete_play_cnt', 'complete_play_user_num',
       'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt',
       'long_time_play_user_num', 'short_time_play_cnt',
       'short_time_play_user_num', 'play_progress', 'comment_stay_duration',
       'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt',
       'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt',
       'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt',
       'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt',
       'comment_like_user_num', 'follow_cnt', 'follow_user_num',
       'cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt',
       'share_user_num', 'download_cnt', 'download_user_num', 'report_cnt',
       'report_user_num', 'reduce_similar_cnt', 'reduce_similar_user_num',
       'collect_cnt', 'collect_user_num', 'cancel_collect_cnt',
       'cancel_collect_user_num']].corr()

upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]
# These columns are dropped as we don't need them anymore 
# (merged in previous cell or just not needed for collaborative-filtering)

to_drop.extend(['date', 'upload_dt', 'video_duration', 'music_id',
                'video_height', 'video_width','video_tag_name',
                'play_progress', 'author_id',  'video_tag_id'
              ])

item_features.drop(columns=to_drop, inplace=True, errors='ignore')

In [22]:
train_df = pd.merge(train_df, item_features, on='video_id', how='left')
test_df = pd.merge(test_df, item_features, on='video_id', how='left')

In [23]:
def build_engagement_score(df):
    """
    Build a comprehensive engagement score based on multiple factors:
    - Watch behavior (watch ratio, complete watches)
    - Active engagement (likes, comments, shares)
    - Content characteristics (video type, visibility, upload type)
    
    This creates a more holistic measure of user engagement with videos.
    """
    # Initialize with base score
    df["engagement_score"] = 0
    
    # 1. Watch behavior - most important signals
    if 'watch_ratio' in df.columns:
        # Watch ratio is a strong signal of interest - scale from 0 to 10
        df["engagement_score"] += df['watch_ratio'].fillna(0) * 10
    
    # 2. Video characteristics
    # Short videos typically have higher engagement
    df["engagement_score"] += df['is_short_video'].fillna(0) * 3
    
    # Video age can affect relevance (newer content may be more relevant)
    if 'video_age' in df.columns:
        # Normalize video age (capped at 365 days)
        max_age = 365
        normalized_age = np.minimum(df['video_age'].fillna(max_age), max_age) / max_age
        # Newer videos get up to 2 points bonus
        df["engagement_score"] += (1 - normalized_age) * 2
    
    # 3. Video metadata
    # Video type (regular content vs ads)
    if 'video_type' in df.columns:
        df["engagement_score"] += np.where(
            df['video_type'] == 'AD',
            -3,  # penalty for ads
            2    # bonus for regular content
        )
    
    # Visibility status
    if 'visible_status' in df.columns:
        df["engagement_score"] += np.where(
            df['visible_status'] == 'public',
            2,   # public videos are more accessible
            -1   # private videos may be less relevant for recommendations
        )
    
    # Upload type - certain formats may be more engaging
    if 'upload_type' in df.columns:
        upload_type_weights = {
            'ShortImport': 3,     # Short imported videos tend to be high quality
            'StartCamera': 2.5,   # Original camera content
            'Knowle': 2,          # Knowledge content
            'Web': 1.5,           # Web content
            'LongImport': 1,      # Long imported videos
            'UNKNOWN': 0,
            'LongCamera': 0.5,
            'PictureSet': 0.5,
            'LongPicture': 0.5,
            'ACurlVideo': 0.5,
            'followShot': 0.5,
            'ShareFromOtherApp': 0.5,
            'SameFrame': 0,
            'PictureCopy': 0,
            'FlashPhoto': 0,
            'PhotoCopy': 0,
            'LocalCollection': 0,
            'LocalInteraction': 0
        }
        df["engagement_score"] += df['upload_type'].map(upload_type_weights).fillna(0)
    
    engagement_columns = {
        'like_cnt': 0.5,      
        'comment_cnt': 0.7,        
        'share_cnt': 0.8,          
        'collect_cnt': 0.6,        
        'follow_cnt': 0.9,        
        'complete_play_cnt': 0.7,  
        'valid_play_cnt': 0.5    
    }
    
    for col, weight in engagement_columns.items():
        if col in df.columns:
            df["engagement_score"] += np.minimum(np.log1p(df[col].fillna(0)) * weight, 10)

    df["engagement_score"] = np.clip(df["engagement_score"], 0, 100)
    
    return df

test_df = build_engagement_score(test_df)
train_df = build_engagement_score(train_df)

In [24]:
to_drop = ['video_type', 'visible_status', 'upload_type']
train_df.drop(columns=to_drop, inplace=True, errors='ignore')
test_df.drop(columns=to_drop, inplace=True, errors='ignore')

In [25]:
def detect_language(text):
    try:
        return detect(text)
    except:
        return "UNKNOWN"
    
captions['language'] = captions['caption'].apply(detect_language)

In [26]:
unique_videos = pd.DataFrame({'video_id': interactions['video_id'].unique()})
captions = pd.merge(unique_videos, captions, on='video_id', how='left')
video_to_caption_idx = {video_id: idx for idx, video_id in enumerate(captions['video_id'])}
del unique_videos

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer

# Load the BERT tokenizers
tokenizer_cn = BertTokenizer.from_pretrained("bert-base-chinese")
tokenizer_kr = BertTokenizer.from_pretrained("beomi/kcbert-base")

# Tokenize the text based on language
def tokenize_by_lang(text, lang):
    text = str(text)
    if lang == 'zh-cn':
        return ' '.join(tokenizer_cn.tokenize(text))
    elif lang == 'ko':
        return ' '.join(tokenizer_kr.tokenize(text))
    else:
        return 'UNKNOWN'

captions['tokenized'] = captions.apply(lambda row: tokenize_by_lang(row['caption'], row['language']), axis=1)

tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(captions['tokenized'])

text_sim = cosine_similarity(tfidf_matrix)

In [28]:
captions.drop_duplicates(subset='video_id', inplace=True)
captions.reset_index(inplace=True)
captions.fillna('UNKNOWN', inplace=True)
captions.drop(columns=['index'], inplace=True, errors='ignore')

/var/folders/pg/1l4d75p90r565jztmcw4v8c00000gn/T/ipykernel_60597/11791697.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'UNKNOWN' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  captions.fillna('UNKNOWN', inplace=True)


In [29]:
video_to_caption_idx = {video_id: idx for idx, video_id in enumerate(captions['video_id'])}

In [30]:
"""
engagement = train_df[['video_id', 'engagement_score']]
captions = pd.merge(captions, engagement, on='video_id', how='left')
"""

"\nengagement = train_df[['video_id', 'engagement_score']]\ncaptions = pd.merge(captions, engagement, on='video_id', how='left')\n"

# Train for ALS

In [31]:
# Get Unique user and videos
user_ids_train = train_df['user_id'].unique()
video_ids_train = train_df['video_id'].unique()

# Compute index for each user and videos
user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids_train)}
video_to_index = {video_id: idx for idx, video_id in enumerate(video_ids_train)}
index_to_video = {idx: video_id for video_id, idx in video_to_index.items()}

# add the index to the train and test
train_df['user_index'] = train_df['user_id'].map(user_to_index)
train_df['video_index'] = train_df['video_id'].map(video_to_index)

test_df['user_index'] = test_df['user_id'].map(user_to_index)
test_df['video_index'] = test_df['video_id'].map(video_to_index)

In [32]:
row = train_df['user_index'].values
col = train_df['video_index'].values

data = train_df['engagement_score'].values

n_users = train_df['user_index'].max() + 1
n_items = train_df['video_index'].max() + 1
    
user_item_matrix = csr_matrix((data, (row, col)), shape=(n_users, n_items))

In [33]:
R = (user_item_matrix != 0).astype(float)

def normalize_ratings(Y, R):
    Ymean = np.zeros(Y.shape[0])
    for i in range(Y.shape[0]):
        if np.sum(R[i, :]) > 0:  # Check if user has any ratings
            Ymean[i] = np.sum(Y[i, :] * R[i, :]) / np.sum(R[i, :])
    
    Ynorm = np.zeros_like(Y)
    for i in range(Y.shape[0]):
        Ynorm[i, :] = (Y[i, :] - Ymean[i]) * R[i, :]
        
    return Ynorm, Ymean

Y_dense = user_item_matrix.toarray()
R_dense = R.toarray()

Ynorm, Ymean = normalize_ratings(Y_dense, R_dense)
user_item_matrix = csr_matrix(Ynorm)

In [34]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(
    factors=15,
    regularization=0.2,
    iterations=15,
    use_gpu=False,
    alpha=10
)

model.fit(user_item_matrix.T) 

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.05292201042175293 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

# Train for Content-Based

In [35]:
# Get the number of videos
n_videos = len(captions)

first_level_sim = (captions['first_level_category_name'].values[:, None] == captions['first_level_category_name'].values).astype(float)

second_level_sim = (captions['second_level_category_name'].values[:, None] == captions['second_level_category_name'].values).astype(float)

third_level_sim = (captions['third_level_category_name'].values[:, None] == captions['third_level_category_name'].values).astype(float)

# Weights for the similarities
text_weight = 0.5
first_level_weight = 0.3
second_level_weight = 0.15
third_level_weight = 0.05

# Combining the similarity matrices with weights
combined_sim = (
    text_sim * text_weight +
    first_level_weight * first_level_sim +
    second_level_weight * second_level_sim +
    third_level_weight * third_level_sim
)

combined_sim = np.clip(combined_sim, 0, 1)

# Metrics

In [ ]:
def precision_at_k(recommended_items, relevant_items, k):
    """Calculate precision@k"""
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    if not recommended_items:
        return 0.0
    
    hit = len(set(recommended_items) & set(relevant_items))
    return hit / min(k, len(recommended_items))

def recall_at_k(recommended_items, relevant_items, k):
    """Calculate recall@k"""
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    if not relevant_items:
        return 0.0
    
    hit = len(set(recommended_items) & set(relevant_items))
    return hit / len(relevant_items)

def ndcg_at_k(recommended_items, relevant_items, k):
    """Calculate nDCG@k"""
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    if not recommended_items or not relevant_items:
        return 0.0
    
    # Create a relevance list where 1 if the item is relevant, 0 otherwise
    relevance = [1 if item in relevant_items else 0 for item in recommended_items]
    
    # Calculate DCG
    dcg = 0
    for i, rel in enumerate(relevance):
        # i+1 because we're using 0-based indexing but rank is 1-based
        dcg += rel / np.log2(i + 2)  # log base 2 of rank+1
    
    # Calculate Ideal DCG (IDCG)
    ideal_relevance = [1] * min(len(relevant_items), k)
    idcg = 0
    for i, rel in enumerate(ideal_relevance):
        idcg += rel / np.log2(i + 2)
    
    return dcg / idcg if idcg > 0 else 0